In [8]:
!pip install transformers datasets evaluate accelerate mlcroissant

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 5.0 MB/s eta 0:00:00
  Created wheel for jsonpath-rw: filename=jsonpath_rw-1.4.0-py3-none-any.whl size=15127 sha256=d216bbc49bbc99a2dde1083da133021b6146672dda6835fdcc6a9a82e7a22445
  Stored in directory: /root/.cache/pip/wheels/e5/8d/50/ee73263c97069bd6040ff40633d444fefaac7beff73abe81a7
Successfully built jsonpath-rw


In [5]:
from huggingface_hub import notebook_login
notebook_login()

In [14]:
from datasets import load_dataset
Medicines_data_Set = load_dataset("kaysss/Medicines")

README.md: 0.00B [00:00, ?B/s]

meds.csv:   0%|          | 0.00/241M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/23939 [00:00<?, ? examples/s]

In [4]:
Medicines_data_Set

DatasetDict({
    train: Dataset({
        features: ['disease_name', 'disease_url', 'med_name', 'med_url', 'final_price', 'price', 'prescription_required', 'drug_varient', 'drug_manufacturer', 'drug_manufacturer_origin', 'drug_content', 'generic_name', 'img_urls'],
        num_rows: 23939
    })
})

In [5]:
import pandas as pd
import numpy as np
len(pd.unique(Medicines_data_Set['train']['disease_name']))
len(pd.unique(Medicines_data_Set['train']['generic_name']))

/tmp/ipykernel_20019/1309903339.py:3: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  len(pd.unique(Medicines_data_Set['train']['disease_name']))
/tmp/ipykernel_20019/1309903339.py:4: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  len(pd.unique(Medicines_data_Set['train']['generic_name']))


6378

In [6]:
DF_P1 = pd.DataFrame(np.transpose([Medicines_data_Set['train']['drug_content'],Medicines_data_Set['train']['generic_name']]),columns=['Descripcion','Medicamento'])

In [7]:
DF_P1 = DF_P1.dropna()
DF_P1 = DF_P1.drop_duplicates()

In [8]:
DF_P1.head()

,Descripcion,Medicamento
0,INTRODUCTION ABOUT ATREST 25MG TABLETATREST 25...,Generic Name Tetrabenazine 25 mg
1,INTRODUCTION ABOUT CAPNEA INJECTIONCAPNEA INJE...,Generic Name Caffeine 20 mg
2,INTRODUCTION ABOUT CAPNEA SOLUTIONCAPNEA SOLUT...,Generic Name Caffeine 20 mg
3,INTRODUCTIONCOGNISTAR 30MG contains Cerebropro...,Generic Name Cerebroprotein Hydrolysate 30 mg
4,INTRODUCTION ABOUT COGNISTAR 60MG INJECTIONCOG...,Generic Name Cerebroprotein Hydrolysate 60 mg


In [9]:
DF_P1["Descripcion"] = DF_P1["Descripcion"].str.replace("INTRODUCTION ABOUT", "", regex=False)
DF_P1["Descripcion"] = DF_P1["Descripcion"].str.strip()
DF_P1["Descripcion"] = DF_P1["Descripcion"].str.replace("INTRODUCTION", "", regex=False)
DF_P1["Descripcion"] = DF_P1["Descripcion"].str.strip()
DF_P1["Medicamento"] = DF_P1["Medicamento"].str.replace("Generic Name", "", regex=False)
DF_P1["Medicamento"] = DF_P1["Medicamento"].str.strip()

In [10]:
DF_P1.head()

,Descripcion,Medicamento
0,ATREST 25MG TABLETATREST 25MG TABLET contains ...,Tetrabenazine 25 mg
1,CAPNEA INJECTIONCAPNEA INJECTION contains Caff...,Caffeine 20 mg
2,CAPNEA SOLUTIONCAPNEA SOLUTION contains Caffei...,Caffeine 20 mg
3,COGNISTAR 30MG contains Cerebroprotein Hydroly...,Cerebroprotein Hydrolysate 30 mg
4,COGNISTAR 60MG INJECTIONCOGNISTAR 60MG INJECTI...,Cerebroprotein Hydrolysate 60 mg


In [11]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
DF_P1["label"] = encoder.fit_transform(DF_P1["Medicamento"])
DF_P1["text"] = DF_P1["Descripcion"]
df_final = DF_P1[["text", "label"]]

In [12]:
df_final["label"].nunique()

6374

In [30]:
min_muestras = 10

conteo_clases = DF_P1["Medicamento"].value_counts()
clases_validas = conteo_clases[conteo_clases >= min_muestras].index
DF_P1_filtrado = DF_P1[DF_P1["Medicamento"].isin(clases_validas)].copy()
DF_P1_filtrado["Medicamento"].nunique(), DF_P1_filtrado.shape

(554, (12036, 4))

In [32]:
encoder = LabelEncoder()

DF_P1_filtrado["label"] = encoder.fit_transform(DF_P1_filtrado["Medicamento"])
id2label = { i: label for i, label in enumerate(encoder.classes_)}
label2id = { label: i for i, label in id2label.items()}
df_final_F = DF_P1_filtrado[["text", "label"]].copy()

In [33]:
from datasets import Dataset, DatasetDict

dataset = Dataset.from_pandas(df_final_F, preserve_index=False)

dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42,
)

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9628
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2408
    })
})

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [15]:
def preprocess_function(examples):
    return tokenizer(examples["text"],truncation=True)

In [38]:
tokenized_dataset = dataset.map(preprocess_function,batched=True)

Map:   0%|          | 0/9628 [00:00<?, ? examples/s]

Map:   0%|          | 0/2408 [00:00<?, ? examples/s]

In [17]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [34]:
num_labels = len(id2label)

In [25]:
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

In [26]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_metric.compute(predictions=predictions,references=labels)
    f1_macro = f1_metric.compute(predictions=predictions,references=labels,average="macro")
    f1_weighted = f1_metric.compute(predictions=predictions,references=labels,average="weighted")

    return {"accuracy": accuracy["accuracy"],"f1_macro": f1_macro["f1"],"f1_weighted": f1_weighted["f1"]}

In [35]:
dataset_sample = DatasetDict({
    "train": dataset["train"].shuffle(seed=42).select(range(1000)),
    "test": dataset["test"].shuffle(seed=42).select(range(1000))
})

In [36]:
tokenized_sample = dataset_sample.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [37]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

learning_rates = [5e-5, 3e-5, 2e-5, 1e-5]

resultados = []

for lr in learning_rates:

    print(f"\nEntrenando con learning_rate = {lr}")

    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert/distilbert-base-uncased",
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir=f"./prueba_lr_{lr}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=lr,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=2,
        weight_decay=0.01,
        logging_steps=50
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_sample["train"],
        eval_dataset=tokenized_sample["test"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    metricas = trainer.evaluate()
    metricas["learning_rate"] = lr

    resultados.append(metricas)


Entrenando con learning_rate = 5e-05


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,6.283165,6.274703,0.003000,0.000014,0.000018
2,6.088118,6.273097,0.003000,0.000014,0.000018



Entrenando con learning_rate = 3e-05


...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,6.289371,6.277084,0.003000,0.000014,0.000018
2,6.145295,6.239159,0.003000,0.000014,0.000018



Entrenando con learning_rate = 1e-05


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,6.304660,6.298384,0.006000,0.000830,0.001847
2,6.228267,6.278620,0.003000,0.000014,0.000018


In [41]:
model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert/distilbert-base-uncased",
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )

training_args = TrainingArguments(
    output_dir="Medicamentos",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,6.156873,5.659923,0.220515,0.065295,0.127904
2,5.515994,4.824507,0.340532,0.152967,0.233153
3,4.867050,4.168951,0.401163,0.209777,0.288457
4,4.301806,3.665375,0.446429,0.265587,0.335258
5,3.489223,3.275409,0.492110,0.317330,0.381268
6,3.183253,2.988771,0.517027,0.346091,0.411327
7,2.971058,2.771153,0.539037,0.375682,0.436828
8,2.791462,2.628017,0.551910,0.390641,0.451652
9,2.648494,2.534159,0.563123,0.404486,0.465749
10,2.524707,2.504941,0.572674,0.414287,0.477823


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=6020, training_loss=3.734243788988487, metrics={'train_runtime': 5487.9888, 'train_samples_per_second': 17.544, 'train_steps_per_second': 1.097, 'total_flos': 1.287951287525376e+16, 'train_loss': 3.734243788988487, 'epoch': 10.0})

In [42]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...amentos/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...amentos/model.safetensors:  56%|#####6    |  152MB /  270MB            

CommitInfo(commit_url='https://huggingface.co/FELIPE960407/Medicamentos/commit/f283229d34fbb0e1fc7ea45ae45ab150004acc0d', commit_message='End of training', commit_description='', oid='f283229d34fbb0e1fc7ea45ae45ab150004acc0d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/FELIPE960407/Medicamentos', endpoint='https://huggingface.co', repo_type='model', repo_id='FELIPE960407/Medicamentos'), pr_revision=None, pr_num=None)

In [43]:
text = "I have a severe acne breakout."

In [45]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="FELIPE960407/Medicamentos")
classifier(text)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'Ayurvedic Medicine', 'score': 0.006096662487834692}]

Resumen

In [22]:
from datasets import load_dataset

dataset = load_dataset("cnn_dailymail", "3.0.0")
dataset

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [23]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [40]:
prefix = "summarize: "

In [41]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):

    inputs = [
        prefix + doc
        for doc in examples["article"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["highlights"],
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [42]:
from datasets import DatasetDict

dataset_sample = DatasetDict({
    "train": dataset["train"].shuffle(seed=42).select(range(4000)),
    "test": dataset["test"].shuffle(seed=42).select(range(200))
})

In [43]:
tokenized_dataset = dataset_sample.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [44]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model="t5-small"
)

In [45]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "t5-small"
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [46]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="Resumen",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    push_to_hub=True,
)

In [47]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.088003,1.966909
2,2.037401,1.959985


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2000, training_loss=2.106046844482422, metrics={'train_runtime': 485.4739, 'train_samples_per_second': 16.479, 'train_steps_per_second': 4.12, 'total_flos': 1082651937865728.0, 'train_loss': 2.106046844482422, 'epoch': 2.0})

In [48]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...Resumen/training_args.bin: 100%|##########| 5.14kB / 5.14kB            

  ...Resumen/model.safetensors:  66%|######6   |  160MB /  242MB            

CommitInfo(commit_url='https://huggingface.co/FELIPE960407/Resumen/commit/3a1201cc6fe7cc38e459aba005cc1d68e7abaf12', commit_message='End of training', commit_description='', oid='3a1201cc6fe7cc38e459aba005cc1d68e7abaf12', pr_url=None, repo_url=RepoUrl('https://huggingface.co/FELIPE960407/Resumen', endpoint='https://huggingface.co', repo_type='model', repo_id='FELIPE960407/Resumen'), pr_revision=None, pr_num=None)

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("FELIPE960407/Resumen")
model = AutoModelForSeq2SeqLM.from_pretrained("FELIPE960407/Resumen")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [11]:
texto = """
The patient was admitted to the hospital after presenting
persistent fever, cough and respiratory complications.
Doctors performed several clinical studies and initiated
antibiotic treatment while monitoring oxygen saturation.
"""

inputs = tokenizer(
    "summarize: " + texto,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = model.generate(
    **inputs,
    max_length=20,
    min_length=5,
    num_beams=4
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The patient was admitted to the hospital after presenting persistent fever, cough and respiratory complications.
